# Section 3: Defining Tools

*Duration: 20 minutes*

---

The agent loop built in Section 2 can evaluate its own retrieval and rewrite queries. But it is still limited to one action: search the vector store. A real agent needs a vocabulary of actions — tools — and a mechanism for choosing the right one.

A tool is any callable function the agent can invoke instead of answering directly. The function does the work. The model reads a description of the function and decides whether to use it. If the description is wrong, the selection will be wrong. Tool definition is a writing problem, not a model problem.

This section implements three tools, wraps them in OpenAI-compatible schemas, and demonstrates that the quality of the tool description directly determines whether the model selects the right tool.

## 3.1 The Three Tools

The agent loop needs exactly three capabilities:

1. **Retrieve from the corpus** — search the vector store for relevant chunks
2. **Calculate** — evaluate mathematical expressions for questions involving stats, probabilities, or modifiers
3. **Decline to answer** — explicitly refuse when the corpus does not contain the information

Each of these becomes a Python function. The function signature and return type are the contract. The model never sees the implementation — it only sees the schema description.

In [ ]:
pip install chromadb -q

In [ ]:
import chromadb
print(f"chromadb version: {chromadb.__version__}")
# Dependency conflict warnings from the Red Hat OpenShift AI environment
# (click, opentelemetry) are expected and can be safely ignored.

## 3.2 Tool 1: RAG Retrieval

The retrieval tool wraps the same ChromaDB query used in previous sections. The difference is structural: it is now a standalone function with a defined interface, not inline code buried in a pipeline.

The function takes a query string and returns the top 3 chunks with their cosine distances. The return value is a dictionary — structured data the agent loop can inspect programmatically.

In [ ]:
import chromadb

def rag_retrieval(query: str) -> dict:
    """
    Search the Basic Fantasy RPG corpus for chunks relevant to the query.
    
    This is the same retrieval operation from Sections 1 and 2, wrapped as
    a callable tool. The agent loop calls this function by name when it 
    decides retrieval is the right action.
    """
    # Connect to the persistent ChromaDB instance built during the Escalation Lab
    chroma_client = chromadb.PersistentClient(path="../prebuilt/chroma_db")
    collection = chroma_client.get_collection("basic_fantasy_corpus")
    
    # Query for the top 3 most similar chunks
    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=["documents", "distances"]
    )
    
    # Return structured data the agent loop can inspect
    chunks = []
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        chunks.append({
            "text": doc,
            "distance": round(dist, 4)
        })
    
    return {
        "tool": "rag_retrieval",
        "query": query,
        "chunks": chunks
    }


# Quick test — this will fail if chroma_db is not in prebuilt/
# That is expected if you are running without the Escalation Lab outputs
try:
    test = rag_retrieval("What is a saving throw?")
    print(f"Retrieved {len(test['chunks'])} chunks")
    print(f"Top chunk distance: {test['chunks'][0]['distance']}")
    print(f"Preview: {test['chunks'][0]['text'][:120]}...")
except Exception as e:
    print(f"Retrieval test skipped: {e}")
    print("This is expected if chroma_db is not available. The tool definition is still valid.")

## 3.3 Tool 2: Calculator

Some questions require arithmetic. "What bonus does a Strength of 16 give?" is a table lookup. But "What is the probability of rolling at least 15 on 1d20?" is a calculation. A retrieval-only agent cannot answer it.

The calculator tool evaluates mathematical expressions safely using Python's `ast` module. It parses the expression into an abstract syntax tree and walks it node by node, rejecting anything that is not a number or a basic arithmetic operator. This is deliberately more restrictive than `eval()` — we do not want the agent to execute arbitrary Python.

In [ ]:
import ast
import operator

# Map AST node types to safe operations
_SAFE_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}


def _safe_eval_node(node):
    """Recursively evaluate an AST node using only safe arithmetic operations."""
    if isinstance(node, ast.Expression):
        return _safe_eval_node(node.body)
    elif isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    elif isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPS:
        left = _safe_eval_node(node.left)
        right = _safe_eval_node(node.right)
        return _SAFE_OPS[type(node.op)](left, right)
    elif isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPS:
        return _SAFE_OPS[type(node.op)](_safe_eval_node(node.operand))
    else:
        raise ValueError(f"Unsupported operation: {ast.dump(node)}")


def calculator(expression: str) -> dict:
    """
    Evaluate a mathematical expression safely.
    
    Uses Python's ast module to parse and evaluate arithmetic expressions
    without calling eval(). Only supports numbers and basic operators:
    +, -, *, /, //, %, **
    
    This tool exists for questions that require computation — probability
    calculations, modifier arithmetic, or stat comparisons that go beyond
    what the corpus contains as text.
    """
    try:
        # Parse the expression into an AST — this validates syntax
        tree = ast.parse(expression, mode="eval")
        # Walk the tree using only safe operations
        result = _safe_eval_node(tree)
        return {
            "tool": "calculator",
            "expression": expression,
            "result": result
        }
    except (ValueError, SyntaxError, TypeError, ZeroDivisionError) as e:
        return {
            "tool": "calculator",
            "expression": expression,
            "error": str(e)
        }


# Test with expressions relevant to tabletop RPG rules
print("Calculator tests:")
print(f"  2 + 3          = {calculator('2 + 3')['result']}")
print(f"  6 / 20         = {calculator('6 / 20')['result']}")      # probability
print(f"  (16 - 10) // 2 = {calculator('(16 - 10) // 2')['result']}")  # ability modifier
print(f"  3 * 1 + 2      = {calculator('3 * 1 + 2')['result']}")    # damage roll

# Verify that unsafe expressions are rejected
unsafe = calculator("__import__('os').system('ls')")
print(f"\n  Unsafe expression rejected: {'error' in unsafe}")

## 3.4 Tool 3: No Answer

The passive pipeline in Section 1 always answered, even when it had no relevant context. The `no_answer` tool gives the agent an explicit way to decline.

This is the simplest tool: it takes no arguments and returns a fixed refusal message. Its power is not in what it does but in what it *prevents*. When the agent selects `no_answer`, it is choosing not to hallucinate. The passive pipeline never had that option.

In [ ]:
def no_answer() -> dict:
    """
    Explicitly decline to answer a question.
    
    The agent selects this tool when it determines that the corpus does
    not contain enough information to produce a correct answer. This is
    the architectural fix for out-of-scope questions: instead of 
    hallucinating from irrelevant context, the agent refuses.
    """
    return {
        "answer": "I do not have enough information to answer this question from the available corpus.",
        "tool": "no_answer"
    }


result = no_answer()
print(f"Tool: {result['tool']}")
print(f"Answer: {result['answer']}")

## 3.5 Tool Schemas: What the Model Actually Sees

The model never calls these Python functions directly. It receives a JSON schema that describes each tool: the name, a natural-language description, and the parameters it accepts. The model reads the description, decides which tool fits the question, and returns a structured tool call. The agent loop then executes the corresponding Python function.

This is the OpenAI-compatible tool definition format. Every field matters, but the `description` field matters most. The model's tool selection is only as good as the description it reads.

In [ ]:
import json

# OpenAI-compatible tool definitions
# The model reads these descriptions to decide which tool to call.
# The descriptions must be precise, specific, and unambiguous.

tool_definitions = [
    {
        "type": "function",
        "function": {
            "name": "rag_retrieval",
            "description": (
                "Search the Basic Fantasy RPG rulebook corpus for text passages "
                "relevant to the query. Use this tool when the question asks about "
                "game rules, character classes, combat mechanics, spells, equipment, "
                "or any topic that would be answered by reading the rulebook. "
                "Returns the 3 most relevant text chunks with similarity scores."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to find relevant rulebook passages"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": (
                "Evaluate a mathematical expression. Use this tool when the question "
                "requires arithmetic, probability calculation, or numeric computation "
                "that cannot be answered by reading text alone. Supports +, -, *, /, "
                "//, %, and ** operators. Example: '(16 - 10) // 2' for an ability "
                "score modifier calculation."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A mathematical expression using numbers and basic arithmetic operators"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "no_answer",
            "description": (
                "Decline to answer the question. Use this tool when the question "
                "asks about something not covered in the Basic Fantasy RPG rulebook "
                "corpus, or when the available context is insufficient to provide a "
                "correct answer. Choosing this tool is better than guessing."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

print("Tool definitions:")
for td in tool_definitions:
    fn = td["function"]
    params = list(fn["parameters"]["properties"].keys())
    print(f"  {fn['name']:<20} params: {params}")
    print(f"  {'':20} desc:   {fn['description'][:80]}...")
    print()

## 3.6 Why the Description Matters

The description is not documentation. It is the decision input. The model reads the description and decides: *does this tool match the question I am trying to answer?*

A vague description leads to wrong tool selection. A precise description leads to correct tool selection. The model does not "understand" the tool. It pattern-matches against the description.

The exercise below demonstrates this directly. We define a tool with a vague description, ask the model to select a tool for a question, and observe which tool it picks. Then we fix the description and run the same question again.

In [ ]:
import os
from openai import OpenAI

# Connect to MaaS endpoint
api_key  = os.environ.get("MAAS_API_KEY")
base_url = os.environ.get("MAAS_BASE_URL")
model_id = os.environ.get("MAAS_MODEL_ID", "granite-3-2-8b-instruct")

client = OpenAI(api_key=api_key, base_url=base_url)

# Define the SAME tools but with a deliberately vague description for rag_retrieval
vague_tools = [
    {
        "type": "function",
        "function": {
            "name": "rag_retrieval",
            "description": "Searches for stuff in documents.",  # <-- vague
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The query"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": (
                "Evaluate a mathematical expression. Use this tool when the question "
                "requires arithmetic, probability calculation, or numeric computation."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A mathematical expression"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "no_answer",
            "description": "Use when you can't answer.",  # <-- also vague
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

# Ask a clear retrieval question
test_question = "What is the saving throw for a 3rd level Fighter against Dragon Breath?"

print("=== VAGUE DESCRIPTIONS ===")
print(f"Question: {test_question}\n")

response = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a rules assistant for Basic Fantasy RPG. Use the available tools to answer questions."},
        {"role": "user", "content": test_question}
    ],
    tools=vague_tools,
    temperature=0.0
)

choice = response.choices[0]
if choice.message.tool_calls:
    for tc in choice.message.tool_calls:
        print(f"  Tool selected : {tc.function.name}")
        print(f"  Arguments     : {tc.function.arguments}")
else:
    print(f"  No tool selected. Model answered directly: {choice.message.content[:150]}")

print(f"\nThe vague description gave the model insufficient signal to select confidently.")

In [ ]:
# Now run the SAME question with the precise descriptions from Section 3.5
print("=== PRECISE DESCRIPTIONS ===")
print(f"Question: {test_question}\n")

response = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a rules assistant for Basic Fantasy RPG. Use the available tools to answer questions."},
        {"role": "user", "content": test_question}
    ],
    tools=tool_definitions,  # <-- the well-described tools from 3.5
    temperature=0.0
)

choice = response.choices[0]
if choice.message.tool_calls:
    for tc in choice.message.tool_calls:
        print(f"  Tool selected : {tc.function.name}")
        print(f"  Arguments     : {tc.function.arguments}")
else:
    print(f"  No tool selected. Model answered directly: {choice.message.content[:150]}")

print(f"\nThe precise description told the model exactly when to use this tool.")

> **Facilitator note:** The vague-vs-precise exercise is the single most important point in this section. If participants take one thing away, it should be this: tool selection quality is a writing problem. The model did not get smarter between the two calls. The description got better.

---

> **FIELD TAKEAWAY**
>
> A tool is a function plus a description. The function does the work. The description determines whether the model selects it. Vague descriptions produce unreliable tool selection. Precise descriptions that specify *when* to use the tool — not just *what* it does — produce reliable selection. If your agent is calling the wrong tool, fix the description before blaming the model.

## 3.7 Saving Tool Definitions

Save the tool definitions and schemas to the prebuilt directory so they can be loaded by subsequent sections without re-running this notebook.

In [ ]:
import json

# Save tool definitions for use in Section 4
output = {
    "metadata": {
        "generated_by": "03_Defining_Tools.ipynb",
        "tools_count": len(tool_definitions),
        "tool_names": [td["function"]["name"] for td in tool_definitions]
    },
    "tool_definitions": tool_definitions
}

output_path = "../prebuilt/tool_definitions.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2)

print(f"Saved {len(tool_definitions)} tool definitions to {output_path}")
for td in tool_definitions:
    fn = td["function"]
    print(f"  - {fn['name']}")

## Pre-Built Output

If the cells above did not execute due to endpoint availability or time constraints, run the cell below to load pre-built results and continue the discussion.

This is expected behavior during a workshop. Use the pre-built outputs without apology and keep the discussion moving.

In [ ]:
USE_PREBUILT = False  # Set to True if live execution was not available

if USE_PREBUILT:
    import json
    
    with open("../prebuilt/tool_definitions.json", "r", encoding="utf-8") as f:
        prebuilt = json.load(f)

    tool_definitions = prebuilt["tool_definitions"]

    print("Loaded pre-built Section 3 outputs")
    print(f"Tool definitions: {len(tool_definitions)}")
    for td in tool_definitions:
        print(f"  - {td['function']['name']}")
    print("\nReady to continue to Section 4.")
else:
    print("Using live results. Ready to continue to Section 4.")

---

## What Comes Next

Section 4 wires these tools to the model and runs the full agent loop. The tools are defined. The schemas are written. Now we connect them to the decision cycle built in Section 2 and run the failing questions through it.

Move to `04_RunningTheAgentLoop/04_Running_the_Agent_Loop.ipynb`.